# Исследование среды и NGU

Эпизодическая новизна, intrinsic reward и ограничения упрощения.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Never Give Up (2020) — алгоритм исследования среды, а не новая версия GRPO. В оригинале сочетаются episodic novelty, lifelong novelty, семейство политик с разным весом intrinsic reward и off-policy recurrent обучение.

Здесь воспроизводим только интуицию эпизодического бонуса через расстояния до памяти состояний. Это не полный NGU и не проверка полного алгоритма. Intrinsic reward не равен успешности задачи: агент может предпочесть новизну полезным действиям. Память эпизода нужно сбрасывать на reset.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
states = torch.tensor([[0., 0.], [1., 0.], [0., 0.], [2., 0.], [1., 0.], [5., 0.]])
memory, bonuses = [], []
for state in states:
    if not memory:
        bonus = 1.0
    else:
        distances = torch.cdist(state[None], torch.stack(memory)).square()[0]
        nearest = distances.topk(min(3, len(memory)), largest=False).values
        similarity = (0.1 / (nearest + 0.1)).sum()
        bonus = float(1 / torch.sqrt(1 + similarity))
    bonuses.append(bonus); memory.append(state)
task_reward = torch.tensor([0., 0., 0., 0., 0., 1.])
mixed = task_reward + 0.2 * torch.tensor(bonuses)
print('novelty:', bonuses); print('mixed reward:', mixed.tolist())
plt.plot(bonuses, color='#d73535', marker='o'); plt.ylabel('illustrative novelty bonus'); plt.show()

## Попробуйте сами

Повторите одно состояние 20 раз. Затем добавьте далёкое состояние. Почему бонус должен зависеть от обучаемого представления, а не сырых пикселей?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [Never Give Up](https://arxiv.org/abs/2002.06038)